# Baseline SARIMA

This notebook is a lightweight SARIMA baseline for Walmart weekly sales forecasting. It does not train one SARIMA model per Store-Dept series. Instead, it:

1. aggregates `Weekly_Sales` by week;
2. trains one compact `SARIMA(1,0,1)(1,0,0,52)` model on total weekly sales;
3. uses explicit annual seasonality with `seasonal_order=(1,0,0,52)`;
4. distributes each weekly forecast back to Store-Dept rows using last-year sales shares;
5. evaluates with the competition WMAE metric.

This is intentionally simple and fast, so it can be used as a classical statistical baseline.

In [1]:
%pip install -q "numpy>=1.24,<3" "pandas>=2.0,<3" "scikit-learn>=1.3,<2" "statsmodels>=0.14,<1" "wandb>=0.19,<1"

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

try:
    import wandb
except Exception:
    wandb = None

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## Configuration

`VALIDATION_WEEKS` controls the final chronological holdout. W&B logging is enabled by default. `RUN_TEST_SUBMISSION` is optional and creates a baseline Kaggle file artifact.

In [3]:
DATA_DIR_CANDIDATES = [
    Path("/content/drive/MyDrive/walmart_competition_data"),
    Path("/content/Walmart-Recruiting---Store-Sales-Forecasting/data"),
    Path("../../../../data"),
    Path("../../../data"),
    Path("data"),
]

VALIDATION_WEEKS = 39
HOLIDAY_WEIGHT = 5.0
SARIMA_ORDER = (1, 0, 1)
SARIMA_SEASONAL_ORDER = (1, 0, 0, 52)
RUN_WANDB = True
WANDB_PROJECT = "Walmart-Recruiting---Store-Sales-Forecasting"
WANDB_RUN_NAME = "SARIMA_Baseline_Aggregate"
WANDB_MODE = "online"
RUN_TEST_SUBMISSION = False
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


def resolve_data_dir(candidates):
    required = ["train.csv", "test.csv", "features.csv", "stores.csv"]
    for candidate in candidates:
        if all((candidate / name).exists() for name in required):
            return candidate
    raise FileNotFoundError(
        "Could not find Walmart data files. Update DATA_DIR_CANDIDATES with the folder containing train.csv, test.csv, features.csv, and stores.csv."
    )


def weighted_mae(y_true, y_pred, is_holiday, holiday_weight=5.0):
    weights = np.where(np.asarray(is_holiday).astype(bool), holiday_weight, 1.0)
    return float(
        np.sum(weights * np.abs(np.asarray(y_true) - np.asarray(y_pred)))
        / np.sum(weights)
    )


DATA_DIR = resolve_data_dir(DATA_DIR_CANDIDATES)
print(f"Using data directory: {DATA_DIR.resolve()}")

Using data directory: /content/drive/MyDrive/walmart_competition_data


In [4]:
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["Date"])
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["Date"])
features = pd.read_csv(DATA_DIR / "features.csv", parse_dates=["Date"])
stores = pd.read_csv(DATA_DIR / "stores.csv")

train = train.sort_values(["Date", "Store", "Dept"]).reset_index(drop=True)
test = test.sort_values(["Date", "Store", "Dept"]).reset_index(drop=True)

print(train.shape, test.shape, features.shape, stores.shape)
display(train.head())

(421570, 5) (115064, 4) (8190, 12) (45, 3)


,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,2,2010-02-05,50605.27,False
2,1,3,2010-02-05,13740.12,False
3,1,4,2010-02-05,39954.04,False
4,1,5,2010-02-05,32229.38,False


## Chronological Validation Split

The split uses the last `VALIDATION_WEEKS` dates as validation. This prevents random leakage across time and matches forecasting behavior better than a random split.

In [5]:
all_dates = np.array(sorted(train["Date"].unique()))
validation_dates = all_dates[-VALIDATION_WEEKS:]
validation_start = validation_dates[0]

train_part = train[train["Date"] < validation_start].copy()
val_part = train[train["Date"] >= validation_start].copy()

print(
    {
        "train_rows": len(train_part),
        "validation_rows": len(val_part),
        "train_end": train_part["Date"].max().date(),
        "validation_start": pd.Timestamp(validation_start).date(),
        "validation_end": val_part["Date"].max().date(),
    }
)

{'train_rows': 305982, 'validation_rows': 115588, 'train_end': datetime.date(2012, 1, 27), 'validation_start': datetime.date(2012, 2, 3), 'validation_end': datetime.date(2012, 10, 26)}


## Aggregate SARIMA Baseline

A full per-series SARIMA approach would require thousands of separate models. For a fast baseline, this notebook trains one SARIMA model on total weekly sales and uses Store-Dept historical proportions to recover row-level predictions.

In [6]:
def weekly_total_sales(frame):
    return (
        frame.groupby("Date", as_index=True)["Weekly_Sales"]
        .sum()
        .sort_index()
        .asfreq("W-FRI")
    )


def fit_aggregate_sarima(train_frame, order=SARIMA_ORDER, seasonal_order=SARIMA_SEASONAL_ORDER):
    weekly = weekly_total_sales(train_frame)
    model = SARIMAX(
        weekly, order=order, seasonal_order=seasonal_order, enforce_stationarity=False, enforce_invertibility=False
    )
    result = model.fit()
    return result, weekly


def make_last_year_share_forecast(target_frame, history_frame, aggregate_forecast):
    target = target_frame[["Store", "Dept", "Date", "IsHoliday"]].copy()
    history = history_frame[["Store", "Dept", "Date", "Weekly_Sales"]].copy()
    history["Date"] = history["Date"] + pd.Timedelta(days=364)
    history = history.rename(columns={"Weekly_Sales": "last_year_sales"})

    series_mean = (
        history_frame.groupby(["Store", "Dept"], as_index=False)["Weekly_Sales"]
        .mean()
        .rename(columns={"Weekly_Sales": "series_mean_sales"})
    )

    target = target.merge(history, on=["Store", "Dept", "Date"], how="left")
    target = target.merge(series_mean, on=["Store", "Dept"], how="left")
    target["allocation_base"] = (
        target["last_year_sales"]
        .fillna(target["series_mean_sales"])
        .fillna(0.0)
        .clip(lower=0.0)
    )

    date_base_sum = target.groupby("Date")["allocation_base"].transform("sum")
    row_count = target.groupby("Date")["allocation_base"].transform("size")
    target["share"] = np.where(
        date_base_sum > 0, target["allocation_base"] / date_base_sum, 1.0 / row_count
    )

    aggregate_map = pd.Series(
        np.asarray(aggregate_forecast), index=sorted(target["Date"].unique())
    )
    target["aggregate_forecast"] = target["Date"].map(aggregate_map)
    target["prediction"] = (
        (target["aggregate_forecast"] * target["share"]).fillna(0.0).clip(lower=0.0)
    )
    return target["prediction"].to_numpy()


def make_seasonal_naive_forecast(target_frame, history_frame):
    target = target_frame[["Store", "Dept", "Date"]].copy()
    history = history_frame[["Store", "Dept", "Date", "Weekly_Sales"]].copy()
    history["Date"] = history["Date"] + pd.Timedelta(days=364)
    history = history.rename(columns={"Weekly_Sales": "last_year_sales"})
    fallback = (
        history_frame.groupby(["Store", "Dept"], as_index=False)["Weekly_Sales"]
        .median()
        .rename(columns={"Weekly_Sales": "series_median_sales"})
    )
    target = target.merge(history, on=["Store", "Dept", "Date"], how="left")
    target = target.merge(fallback, on=["Store", "Dept"], how="left")
    return (
        target["last_year_sales"]
        .fillna(target["series_median_sales"])
        .fillna(0.0)
        .clip(lower=0.0)
        .to_numpy()
    )

In [7]:
sarima_result, train_weekly = fit_aggregate_sarima(train_part, SARIMA_ORDER, SARIMA_SEASONAL_ORDER)
aggregate_val_forecast = sarima_result.forecast(steps=VALIDATION_WEEKS).clip(lower=0.0)

val_pred = make_last_year_share_forecast(val_part, train_part, aggregate_val_forecast)
seasonal_naive_pred = make_seasonal_naive_forecast(val_part, train_part)

validation_wmae = weighted_mae(
    val_part["Weekly_Sales"], val_pred, val_part["IsHoliday"], HOLIDAY_WEIGHT
)
validation_mae = mean_absolute_error(val_part["Weekly_Sales"], val_pred)
validation_rmse = np.sqrt(mean_squared_error(val_part["Weekly_Sales"], val_pred))
seasonal_naive_wmae = weighted_mae(
    val_part["Weekly_Sales"], seasonal_naive_pred, val_part["IsHoliday"], HOLIDAY_WEIGHT
)

metrics = {
    "validation/wmae": validation_wmae,
    "validation/mae": float(validation_mae),
    "validation/rmse": float(validation_rmse),
    "baseline/seasonal_naive_wmae": seasonal_naive_wmae,
    "improvement_vs_seasonal_naive_pct": 100.0
    * (seasonal_naive_wmae - validation_wmae)
    / seasonal_naive_wmae,
    "sarima_order": str(SARIMA_ORDER),
    "sarima_seasonal_order": str(SARIMA_SEASONAL_ORDER),
}

metrics

{'validation/wmae': 1856.8605253613277,
 'validation/mae': 1843.2876309305175,
 'validation/rmse': 3919.5943699723775,
 'baseline/seasonal_naive_wmae': 1800.1735916704909,
 'improvement_vs_seasonal_naive_pct': -3.148970407805707,
 'sarima_order': '(1, 1, 1)'}

In [8]:
if RUN_WANDB and wandb is None:
    raise ImportError(
        "wandb is not available. Re-run the install/import cell or install wandb."
    )

if RUN_WANDB:
    wandb_metrics = {
        key: float(value)
        for key, value in metrics.items()
        if isinstance(value, (int, float, np.integer, np.floating))
    }
    run = wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        job_type="baseline-validation",
        mode=WANDB_MODE,
        reinit=True,
        config={
            "model": "aggregate_sarima_last_year_share",
            "sarima_order": SARIMA_ORDER,
            "sarima_seasonal_order": SARIMA_SEASONAL_ORDER,
            "validation_weeks": VALIDATION_WEEKS,
            "holiday_weight": HOLIDAY_WEIGHT,
            "allocation": "last_year_sales_share_with_series_mean_fallback",
            "data_dir": str(DATA_DIR),
        },
    )
    wandb.log(wandb_metrics)
    wandb.summary.update(wandb_metrics)
    wandb.summary["sarima_order"] = str(SARIMA_ORDER)
    wandb.summary["sarima_seasonal_order"] = str(SARIMA_SEASONAL_ORDER)
    wandb.summary["validation_start"] = str(pd.Timestamp(validation_start).date())
    wandb.summary["validation_end"] = str(val_part["Date"].max().date())
    wandb.finish()
    print(f"Logged W&B run: {WANDB_RUN_NAME}")
else:
    print("W&B logging skipped because RUN_WANDB=False.")

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nmetr23 (kende23-n-a) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


baseline/seasonal_naive_wmae,▁
improvement_vs_seasonal_naive_pct,▁
validation/mae,▁
validation/rmse,▁
validation/wmae,▁
baseline/seasonal_naive_wmae,1800.17359
improvement_vs_seasonal_naive_pct,-3.14897
sarima_order,"(1, 1, 1)"
validation/mae,1843.28763
validation/rmse,3919.59437
validation/wmae,1856.86053


Logged W&B run: SARIMA_Baseline_Aggregate
